In [7]:
import io
import json
import os
from typing import Callable

import polars as pl
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

API_KEY = os.getenv("GEMINI_API_KEY")

# MODEL = "gemini-3.1-flash-lite-preview"
MODEL = "gemini-2.5-flash-lite"


client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)


In [8]:
def make_llm_request(prompt: str) -> str:
    client = OpenAI(
        api_key=API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    messages = [
        {"role": "developer", "content": "You are a weather assistant."},
        {"role": "user", "content": prompt},
    ]

    tool_definitions, tool_name_to_func = get_tool_definitions()

    # guard: loop limit, we break as soon as we get an answer
    for _ in range(10):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tool_definitions,  # always pass all tools in this example
            tool_choice="auto",
            max_completion_tokens=1000,
            # extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        resp_message = response.choices[0].message
        messages.append(
            {k: v for k, v in resp_message.model_dump().items() if v is not None}
        )

        print(f"Generated message: {resp_message.model_dump()}")
        print()

        # parse possible tool calls (assume only "function" tools)
        if resp_message.tool_calls:
            for tool_call in resp_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # call tool, serialize result, append to messages
                func = tool_name_to_func[func_name]
                func_result = func(**func_args)

                messages.append(
                    {
                        "role": "tool",
                        "content": json.dumps(func_result),
                        "tool_call_id": tool_call.id,
                    }
                )
        else:
            # no tool calls, we're done
            return resp_message.content

    # we should not get here
    last_response = resp_message.content
    return f"Could not resolve request, last response: {last_response}"


def get_tool_definitions() -> tuple[list[dict], dict[str, Callable]]:
    tool_definitions = [
        {
            "type": "function",
            "function": {
                "name": "read_remote_csv",
                "description": "Download a CSV file from a URL and return its contents as text.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "The URL of the CSV file to read.",
                        },
                    },
                    "required": ["url"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "read_remote_parquet",
                "description": "Download a Parquet file from a URL and return its contents as text.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "The URL of the Parquet file to read.",
                        },
                    },
                    "required": ["url"],
                },
            },
        },
    ]

    tool_name_to_callable = {
        "read_remote_csv": read_remote_csv_tool,
        "read_remote_parquet": read_remote_parquet_tool,
    }

    return tool_definitions, tool_name_to_callable


def read_remote_csv_tool(url: str) -> str:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    df = pl.read_csv(io.BytesIO(response.content))
    return df.head(20).write_csv()


def read_remote_parquet_tool(url: str) -> str:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    df = pl.read_parquet(io.BytesIO(response.content))
    return df.head(20).write_csv()


Csv call: 

In [3]:
prompt = "Based on returned data, tell me the domain of this dataset: https://github.com/j-adamczyk/ApisTox_dataset/blob/master/outputs/dataset_final.csv"
response = make_llm_request(prompt)
print("Prompt:\n", prompt)
print("Response:\n", response)


Generated message: {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'i9ywlzwh', 'function': {'arguments': '{"url":"https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv"}', 'name': 'read_remote_csv'}, 'type': 'function', 'extra_content': {'google': {'thought_signature': 'EjQKMgEMOdbHlmHdKAy0FDzNLy079WD5QNqb9fWV5Kq2C72d+rdoXNM/aRJFsSgXPA1aSp3+'}}}]}

Generated message: {'content': 'The domain of this dataset is **ecotoxicology**, specifically focusing on the **toxicity of chemical compounds (agrochemicals) to bees (Apis mellifera)**.\n\nThe dataset contains information about:\n*   **Chemical identifiers:** Names, CIDs, CAS numbers, and SMILES strings.\n*   **Agrochemical classifications:** Indicates whether a compound is an herbicide, fungicide, or insecticide.\n*   **Toxicity data:** Specifies the type of exposure (e.g., Oral, Contact) and a label f

Parquet call: 

In [ ]:
prompt = "Tell me something interesting about this dataset: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
response = make_llm_request(prompt)
print("Prompt:\n", prompt)
print("Response:\n", response)

print()

Generated message: {'content': "I'm sorry, but I can only access publicly available data. The URL you provided appears to be a webpage with information about the dataset rather than a direct link to the data file itself. Could you please provide a direct link to a CSV or Parquet file that I can access?", 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}

Prompt:
 Tell me something interesting about this dataset: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page#:~:text=January-,Yellow%20Taxi%20Trip%20Records,-(PARQUET)
Response:
 I'm sorry, but I can only access publicly available data. The URL you provided appears to be a webpage with information about the dataset rather than a direct link to the data file itself. Could you please provide a direct link to a CSV or Parquet file that I can access?

